# Table 2 Analysis

This notebook parses the experiment log files for the **Hybrid Coordinate**, **Hybrid Standard**, and **Primal TF** models.

From the results, we compute:
- **Success Rate**
- **Average Steps per Episode**
- **Collision Rate** (Total Collisions / Total Steps)



In [33]:
import re
import pandas as pd
from pathlib import Path

In [34]:
base_path = Path("Results_table_2")

files = {
    "Hybrid Coordinate": base_path / "Hybrid_coordinate_successrate.txt",
    "Hybrid Standard": base_path / "Hybrid_standard_successrate.txt",
    "Primal TF": base_path / "Primal_TF__successrate.txt"
}

In [35]:
import re

def parse_results(file_path):
    successes = 0
    total = 0
    steps_list = []

    with open(file_path, "r") as f:
        for line in f:
            if "Success:" in line:

                total += 1

                # success
                if "Success: True" in line:
                    successes += 1

                # extract steps
                step_match = re.search(r"Steps:\s*(\d+)", line)
                if step_match:
                    steps_list.append(int(step_match.group(1)))

    success_rate = successes / total if total > 0 else 0
    avg_steps = sum(steps_list) / len(steps_list) if steps_list else 0

    return {
        "total_experiments": total,
        "successful": successes,
        "success_rate": success_rate,
        "avg_steps": avg_steps
    }

In [36]:
results = []

for model, file in files.items():
    stats = parse_results(file)
    stats["model"] = model
    results.append(stats)

df = pd.DataFrame(results)
df

,total_experiments,successful,success_rate,avg_steps,model
0,336,113,0.336310,206.470238,Hybrid Coordinate
1,336,214,0.636905,146.666667,Hybrid Standard
2,336,49,0.145833,231.547619,Primal TF


In [37]:
df["success_rate_percent"] = df["success_rate"] * 100
df

,total_experiments,successful,success_rate,avg_steps,model,success_rate_percent
0,336,113,0.336310,206.470238,Hybrid Coordinate,33.630952
1,336,214,0.636905,146.666667,Hybrid Standard,63.690476
2,336,49,0.145833,231.547619,Primal TF,14.583333


# Now we calcluate the collision rate of PRIMAl TF version

In [38]:
import re

primal_file = "Results_table_2/Primal_TF__successrate.txt"

def collision_rate_from_steps(file_path):

    total_collisions = 0
    total_steps = 0
    total_episodes = 0

    with open(file_path, "r") as f:
        for line in f:

            if line.startswith("[EPISODE"):

                total_episodes += 1

                step_match = re.search(r"Steps:\s*(\d+)", line)
                collision_match = re.search(r"Collisions:\s*([0-9.]+)", line)

                if step_match:
                    total_steps += int(step_match.group(1))

                if collision_match:
                    total_collisions += float(collision_match.group(1))

    collision_rate = total_collisions / total_steps if total_steps > 0 else 0

    return total_episodes, total_steps, total_collisions, collision_rate


episodes, steps, collisions, rate = collision_rate_from_steps(primal_file)

print(f"Total Episodes: {episodes}")
print(f"Total Steps: {steps}")
print(f"Total Collisions: {collisions}")
print(f"Collision Rate (collisions/step): {rate:.6f}")
print(f"Collision Rate: {rate*100:.4f}%")

Total Episodes: 336
Total Steps: 77800
Total Collisions: 68.15150793650805
Collision Rate (collisions/step): 0.000876
Collision Rate: 0.0876%


# Now we calcluate the collision rate of Both Hybrid standard and cordinate

In [39]:
import re
import pandas as pd

files = {
    "Hybrid Coordinate": "Results_table_2/Hybrid_coordinate_collisionrate.txt",
    "Hybrid Standard": "Results_table_2/Hybrid_standard_collisionrate.txt"
}

def compute_collision_rate(file_path):

    total_steps = 0
    total_collisions = 0
    episodes = 0

    with open(file_path, "r") as f:
        for line in f:

            if line.startswith("[EPISODE"):

                episodes += 1

                step_match = re.search(r"Steps:\s*(\d+)", line)
                collision_match = re.search(r"Collisions:\s*([0-9.]+)", line)

                if step_match:
                    total_steps += int(step_match.group(1))

                if collision_match:
                    total_collisions += float(collision_match.group(1))

    collision_rate = total_collisions / total_steps if total_steps > 0 else 0

    return {
        "episodes": episodes,
        "total_steps": total_steps,
        "total_collisions": total_collisions,
        "collision_rate": collision_rate
    }


results = []

for model, file in files.items():
    stats = compute_collision_rate(file)
    stats["model"] = model
    results.append(stats)

df = pd.DataFrame(results)
df

,episodes,total_steps,total_collisions,collision_rate,model
0,14,1397,107.200000,0.076736,Hybrid Coordinate
1,49,4790,63.813333,0.013322,Hybrid Standard
